In [1]:
# Install the packages (run once).
!pip install -q -U transformers accelerate peft trl
# datasets and pandas are installed separately and not upgraded: Google Colab needs the versions it ships with
!pip install -q datasets pandas
# Google Colab ships an old version of torchao that the current peft rejects. We do not use it, so remove it.
!pip uninstall -y -q torchao

**Packages used in this tutorial**

| Package | What it does | Where we use it |
|---|---|---|
| `transformers` | Loads models and tokenizers; generates text | `pipeline`, `AutoModelForCausalLM`, `AutoTokenizer` |
| `accelerate` | Places the model on the available device (GPU or CPU) | needed for `device_map="auto"` |
| `datasets` | Downloads and handles datasets from the Hugging Face hub | `load_dataset`, `Dataset.from_list` |
| `peft` | Parameter-efficient fine-tuning methods, including LoRA | `LoraConfig` |
| `trl` | Trainer for supervised fine-tuning of chat models | `SFTTrainer`, `SFTConfig` |
| `pandas` | Tables | the final comparison table |

PyTorch is installed automatically as a dependency. All of these are open-source libraries maintained by Hugging Face, except `pandas`.

In [2]:
from transformers import pipeline

# The model's name on the Hugging Face hub: a small chat model with 0.5 billion parameters
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# Download the model and its tokenizer, and bundle them into one ready-to-use object.
# "text-generation" is the task; device_map="auto" picks a GPU if there is one, otherwise the CPU.
# Nothing is trained here: we only load the model.
pipe = pipeline("text-generation", model=MODEL, device_map="auto")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

## 1.2 Data: MUC-4

MUC-4 is a classic human-annotated dataset for event extraction, created for the Fourth Message Understanding Conference (1992). It contains 1,700 news reports about political violence in Latin America. For each report, annotators filled in a template for every event:

- **incident type**: attack, bombing, kidnapping, arson, robbery, or forced work stoppage
- **roles**: `PerpInd` (individual perpetrators), `PerpOrg` (organisations responsible), `Target` (objects attacked), `Victim` (people harmed), `Weapon`

A report that describes no such event gets an empty list, `[]`. The text is lowercased, as in the original release.

We load a version from the Hugging Face hub (`fnl-es/muc4-chat`) that is already in chat format: the instruction, the report, and the human annotation as the assistant's reply. To keep things fast we keep the shorter reports and sample 100 for training and 30 for testing.

In [3]:
# Get the helper file if it is not already in this folder (needed on Google Colab)
import os, urllib.request
if not os.path.exists("muc4_utils.py"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/jhxu0107-collab/soda_502/main/muc4_utils.py", "muc4_utils.py")

# Our helper functions: load_muc4, generate, score, gold_of
from muc4_utils import *

# Sample 100 short reports for training and 30 for testing (always the same ones)
train, test = load_muc4(n_train=100, n_test=30)
print(len(train), "training reports,", len(test), "test reports")

# Pick a test report that contains at least one event
example = next(ex for ex in test if gold_of(ex) != "[]")

# Each example is a conversation with three messages:
#   system    = the instruction
#   user      = the news report
#   assistant = the human annotation (the answer we want)
for m in example["messages"]:
    print("-----", m["role"])
    print(m["content"])

README.md:   0%|          | 0.00/2.22k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.88MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/dev-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  632kB            

data/dev-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  596kB            

... (3 outputs left out)

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1299 [00:00<?, ? examples/s]

Filter:   0%|          | 0/200 [00:00<?, ? examples/s]

100 training reports, 30 test reports
----- system
Extract every terrorist event described in the document.

Event types: attack, bombing, kidnapping, arson, robbery, forced work stoppage.
Roles: PerpInd (individuals who carried out the event), PerpOrg (organisations responsible), Target (physical objects attacked or damaged), Victim (people killed, injured, kidnapped or otherwise harmed), Weapon (weapons or explosives used).
Military clashes between armed forces are not events.

Answer only with a JSON array of events in order of appearance, or [] if there is none. An event is an object with "incident_type" and its non-empty roles; a role is a list of entities; an entity is a list of every string in the document referring to it, copied verbatim. Example:
[{"incident_type":"bombing","PerpInd":[["guerrillas","the rebels"]],"Target":[["bridge"]]}]
----- user
early in the morning of 21 december fmln rebels raided san jose villanueva, south of san salvador, la libertad department, where th

## 1.3 Zero-shot: prompt only

We give the model the first two messages (instruction and report) and let it write the third. No training.

## 1.4 Score on the test set

We compare the model's answers with the human annotation in four ways, from lenient to strict:

| Measure | Meaning |
|---|---|
| `valid_json` | share of outputs that can be parsed as a JSON list of events |
| `event_types` | share of reports where the incident types are exactly right |
| `role_f1` | F1 over role fillers (perpetrators, targets, victims, weapons); a filler counts as correct if it matches one of the annotated mentions |
| `exact_match` | share of reports where the whole answer is identical to the annotation |

In [4]:
# Now all 30 test reports (this takes a while on a laptop)
raw_outputs = generate(pipe, test)

# Compare with the human annotation. Keep the result: Part 2 compares against it.
baseline = score(raw_outputs, test)
baseline

{'valid_json': 0.6333333333333333,
 'event_types': 0.03333333333333333,
 'role_f1': 0.09090909090909093,
 'exact_match': 0.0}

## 1.5 Takeaways

- The model understands the task without any training: it writes events with the right kind of fields.
- The format is unreliable. About a quarter of the answers are not valid JSON, so they cannot be used at all.
- The content is weak: `event_types` and `role_f1` are low, and no report is answered exactly right.
- The model never answers `[]`, although 7 of the 30 test reports contain no event. It always finds something to report.

Can 100 annotated examples fix this? That is Part 2.

# Part 2: Fine-tuning with LoRA

Same model, same test set. We first train on annotated reports from our training sample, then score again.

## 2.1 Fine-tuning and LoRA

Fine-tuning continues training the model on our own examples, so that it assigns a higher probability to the annotated answers. LoRA keeps all the original weights frozen and trains a small pair of matrices added next to selected weight matrices; `r` sets their size and `lora_alpha` how strongly they are applied.

## 2.2 Prepare the training data

A training example is the full conversation: instruction + news report + human annotation. The dataset already comes in this form.

One choice matters here. In 41 of our 100 training reports the correct answer is `[]` (no event). If we train on all of them, the model learns the easy rule "answer `[]`" and stops extracting. So we keep every report that has events and only 10 of the reports without.

In [5]:
from datasets import Dataset

# Split the training sample: reports with events, and reports whose correct answer is []
with_events = [ex for ex in train if gold_of(ex) != "[]"]
without_events = [ex for ex in train if gold_of(ex) == "[]"]

# Keep all reports with events, and only 10 of the reports without
train_subset = with_events + without_events[:10]
print(len(with_events), "with events +", len(train_subset) - len(with_events), "without =", len(train_subset))

# Training data: the full conversations, including the correct answer
train_ds = Dataset.from_list([{"messages": ex["messages"]} for ex in train_subset])

# Look at the first training example
train_ds[0]["messages"]

59 with events + 10 without = 69


[{'content': 'Extract every terrorist event described in the document.\n\nEvent types: attack, bombing, kidnapping, arson, robbery, forced work stoppage.\nRoles: PerpInd (individuals who carried out the event), PerpOrg (organisations responsible), Target (physical objects attacked or damaged), Victim (people killed, injured, kidnapped or otherwise harmed), Weapon (weapons or explosives used).\nMilitary clashes between armed forces are not events.\n\nAnswer only with a JSON array of events in order of appearance, or [] if there is none. An event is an object with "incident_type" and its non-empty roles; a role is a list of entities; an entity is a list of every string in the document referring to it, copied verbatim. Example:\n[{"incident_type":"bombing","PerpInd":[["guerrillas","the rebels"]],"Target":[["bridge"]]}]',
  'role': 'system'},
 {'content': 'medellin -- four men who were travelling in a privately owned blue renault with license plates le-7570, have set fire to two government

## 2.3 Model + LoRA config

We load a fresh copy of the model for training, and keep the one from Part 1 untouched.

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig

# Load the model and tokenizer again, this time as separate objects for training
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# Number format for training. The model's default is the compact bfloat16 format.
# Older GPUs, such as the free T4 on Google Colab, cannot train in it, so there we use standard float32.
import torch
use_bf16 = not torch.cuda.is_available() or torch.cuda.get_device_capability()[0] >= 8

model = AutoModelForCausalLM.from_pretrained(MODEL, device_map="auto",
                                             dtype=torch.bfloat16 if use_bf16 else torch.float32)
tokenizer = AutoTokenizer.from_pretrained(MODEL)

# LoRA settings
lora_config = LoraConfig(
    r=16,                 # size of the add-on matrices
    lora_alpha=32,        # how strongly the add-on is applied
    lora_dropout=0.05,    # a little regularisation during training
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],   # where to add LoRA: the attention matrices
    task_type="CAUSAL_LM",                                     # the model generates text
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

## 2.4 Train

`SFTTrainer` handles the rest for us: applying the chat template, tokenizing, padding, computing the loss, and updating the LoRA parameters.

In [7]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    peft_config=lora_config,          # adds LoRA and freezes the original weights
    args=SFTConfig(
        output_dir="qwen-muc4-lora",
        num_train_epochs=5,               # go through the training examples five times
        per_device_train_batch_size=2,    # 2 examples at a time, to keep memory use low
        gradient_accumulation_steps=4,    # update after every 4 batches, so 8 examples per update
        learning_rate=2e-4,               # how big each update is
        assistant_only_loss=True,         # compute the loss only on the answer, not on the instruction and report
        bf16=use_bf16,                    # number format, chosen above to suit the GPU
        logging_steps=5,                  # print the loss every 5 steps
        save_strategy="no",               # do not save checkpoints during training
        report_to="none",                 # no external logging service
    ),
)

# How many parameters will actually be trained? Only a small fraction.
trainer.model.print_trainable_parameters()

/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/69 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/69 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/69 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/69 [00:00<?, ? examples/s]

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [8]:
# Train. The loss should go down as the model learns to reproduce the annotated answers.
trainer.train()

Step,Training Loss
5,0.625731
10,0.482593
15,0.376738
20,0.374973
25,0.300975
30,0.267560
35,0.249271
40,0.229495
45,0.235233


TrainOutput(global_step=45, training_loss=0.3491743591096666, metrics={'train_runtime': 198.7491, 'train_samples_per_second': 1.736, 'train_steps_per_second': 0.226, 'total_flos': 418955224934400.0, 'train_loss': 0.3491743591096666, 'epoch': 5.0})

**Reading the output**

- Only about 2.2 million of the model's 496 million parameters are trained (0.44%). The rest stay exactly as they were.
- The loss falls steadily, from about 0.6 to about 0.2. This tells us the model is getting better at reproducing the *training* answers. It does not tell us how well it does on new reports: for that we need the test set.

## 2.5 Score on the same test set

In [9]:
# Wrap the fine-tuned model in a pipeline, so we can use the same generate() as in Part 1
# Switch the model from training mode to evaluation mode before generating.
# Training leaves it in training mode, and text generation does not work correctly there.
trainer.model.eval()

ft_pipe = pipeline("text-generation", model=trainer.model, tokenizer=tokenizer)

In [10]:
import pandas as pd

# Score the fine-tuned model on the same 30 test reports
finetuned = score(generate(ft_pipe, test), test)

# Side by side with the zero-shot scores from Part 1 (needs `baseline` from section 1.4)
pd.DataFrame({"Zero-shot": baseline, "LoRA fine-tuned": finetuned}).round(2)

,Zero-shot,LoRA fine-tuned
valid_json,0.63,0.97
event_types,0.03,0.23
role_f1,0.09,0.22
exact_match,0.00,0.20


In [11]:
# Save the LoRA adapter (a small file); the original model is unchanged
trainer.save_model("qwen-muc4-lora")

## 2.6 Takeaways

- **Format is learned quickly.** After fine-tuning, almost every answer is valid JSON (0.97, up from 0.77).
- **Content improves only a little.** `role_f1` goes up (from 0.13 to 0.22 in our run), but most role fillers are still missed. A small model and about 70 examples are not enough for this task.
- **Check your measures against a simple reference.** `event_types` and `exact_match` go up mostly because the model now answers `[]` more often. Answering `[]` for every report would score 0.23 on both of those measures (see `always_empty`), about the same as the fine-tuned model, and 0 on `role_f1`. Only `role_f1` shows real extraction.
- **The training data shapes the behaviour.** Trained on all 100 reports, the model answered `[]` almost every time. What you put in the training set matters as much as the settings.
- **Keep the test set separate.** Never train on it, and do not tune prompts or settings on it. With only 30 test reports, small differences in scores can be chance.
- **When is fine-tuning worth it?** When you have a fixed task, a few hundred or more annotated examples, and a prompt alone does not give the format or quality you need.

For extension: choose one of the following questions.

**Training set size**. Fine-tune the model on training sets of different sizes and plot role_f1 against the number of training examples. How does the amount of training data affect performance?

**LoRA configuration.** Fine-tune the model with different LoRA settings and record the results. How do the settings affect performance?